# Part 2 — Cross-Domain Acne Classification (Google Colab)

**Before running:** Runtime → Change runtime type → **A100 GPU**

This notebook covers all of Part 2 end-to-end:
1. Setup (clone repo, install deps, download ACNE04)
2. Patch extraction — crop positive/negative patches from ACNE04 bounding boxes
3. Train EfficientNet-B0 classifier on ACNE04 patches
4. Download & prepare DermNet dataset
5. Evaluate on DermNet test set (Accuracy, F1, AUROC)
6. Grad-CAM visualizations on DermNet predictions
7. Reflection

Run cells top to bottom.

---
## Section 1 — Setup

In [ ]:
# Clone repo
import os

REPO_DIR = "/content/AcneDetection"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/EvxLee/AcneDetection.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
    print("Repo already exists — pulled latest.")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Install dependencies
!pip install -q roboflow python-dotenv timm grad-cam scikit-learn
print("Dependencies installed.")

In [ ]:
# Set Roboflow credentials — paste your API key here
import os

os.environ["ROBOFLOW_API_KEY"]   = "YOUR_API_KEY_HERE"   # ← paste your key
os.environ["ROBOFLOW_WORKSPACE"] = "evan-lee-rrndd"
os.environ["ROBOFLOW_PROJECT"]   = "acne04-detection-p8j0d"
os.environ["ROBOFLOW_VERSION"]   = "1"

with open(f"{REPO_DIR}/.env", "w") as f:
    for k in ["ROBOFLOW_API_KEY", "ROBOFLOW_WORKSPACE", "ROBOFLOW_PROJECT", "ROBOFLOW_VERSION"]:
        f.write(f"{k}={os.environ[k]}\n")
print("Credentials set.")

In [ ]:
# Download ACNE04 dataset
!python part1_detection/roboflow_loader.py --download

In [ ]:
# Verify GPU
import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
else:
    print("No GPU — go to Runtime → Change runtime type → A100")

import json
import random
import numpy as np
from pathlib import Path
from PIL import Image

DATA_DIR   = Path("data/acne04")
PATCH_DIR  = Path("data/patches")
PATCH_SIZE = 224
MIN_BOX    = 20
NEG_SIZE   = 90
SEED       = 42

SPLITS = {"train": "train", "valid": "val"}

random.seed(SEED)

In [ ]:
def is_valid_patch(patch_img, min_brightness=40, min_color_spread=15):
    """Reject dark/greyscale patches (backgrounds, hair, watermarks)."""
    arr = np.array(patch_img, dtype=float)
    if arr.mean() < min_brightness:
        return False
    channel_means = arr.mean(axis=(0, 1))
    if channel_means.max() - channel_means.min() < min_color_spread:
        return False
    return True

def has_overlap(box, gt_boxes):
    for g in gt_boxes:
        if box[0] < g[2] and box[2] > g[0] and box[1] < g[3] and box[3] > g[1]:
            return True
    return False

def sample_negative(img, img_w, img_h, gt_boxes, size, max_tries=150):
    for _ in range(max_tries):
        x1 = random.randint(0, max(0, img_w - size))
        y1 = random.randint(0, max(0, img_h - size))
        candidate = [x1, y1, x1 + size, y1 + size]
        if has_overlap(candidate, gt_boxes):
            continue
        patch = img.crop(candidate).resize((PATCH_SIZE, PATCH_SIZE), Image.BILINEAR)
        if is_valid_patch(patch):
            return candidate, patch
    return None, None

def extract_patches(acne04_split, patch_split):
    pos_dir = PATCH_DIR / patch_split / "acne"
    neg_dir = PATCH_DIR / patch_split / "no_acne"
    pos_dir.mkdir(parents=True, exist_ok=True)
    neg_dir.mkdir(parents=True, exist_ok=True)

    with open(DATA_DIR / acne04_split / "_annotations.coco.json") as f:
        coco = json.load(f)

    ann_map = {}
    for ann in coco["annotations"]:
        ann_map.setdefault(ann["image_id"], []).append(ann)

    pos_count = neg_count = skipped = rejected = 0

    for meta in coco["images"]:
        img_id = meta["id"]
        anns   = ann_map.get(img_id, [])
        if not anns:
            continue

        img      = Image.open(DATA_DIR / acne04_split / meta["file_name"]).convert("RGB")
        img_w, img_h = img.size
        gt_boxes = []

        for ann in anns:
            x, y, w, h = ann["bbox"]
            if w < MIN_BOX or h < MIN_BOX:
                skipped += 1
                continue
            x1, y1 = int(x), int(y)
            x2, y2 = min(int(x + w), img_w), min(int(y + h), img_h)
            gt_boxes.append([x1, y1, x2, y2])
            patch = img.crop((x1, y1, x2, y2)).resize((PATCH_SIZE, PATCH_SIZE), Image.BILINEAR)
            ann_id = ann["id"]
            patch.save(pos_dir / f"{img_id}_{ann_id}.jpg", quality=90)
            pos_count += 1

        for i in range(len(gt_boxes)):
            box, patch = sample_negative(img, img_w, img_h, gt_boxes, NEG_SIZE)
            if patch is None:
                rejected += 1
                continue
            patch.save(neg_dir / f"{img_id}_neg{i}.jpg", quality=90)
            neg_count += 1

    print(f"[{acne04_split}]  acne={pos_count}  no_acne={neg_count}  skipped(tiny)={skipped}  rejected(bad)={rejected}")

print("Starting patch extraction...")
for acne04_split, patch_split in SPLITS.items():
    extract_patches(acne04_split, patch_split)
print("Done.")

In [ ]:
def has_overlap(box, gt_boxes):
    """Return True if box overlaps any box in gt_boxes (all in x1y1x2y2)."""
    for g in gt_boxes:
        if box[0] < g[2] and box[2] > g[0] and box[1] < g[3] and box[3] > g[1]:
            return True
    return False

def sample_negative(img_w, img_h, gt_boxes, size, max_tries=100):
    """Sample a random crop of `size` x `size` with zero overlap with gt_boxes."""
    for _ in range(max_tries):
        x1 = random.randint(0, max(0, img_w - size))
        y1 = random.randint(0, max(0, img_h - size))
        candidate = [x1, y1, x1 + size, y1 + size]
        if not has_overlap(candidate, gt_boxes):
            return candidate
    return None  # image too crowded to find a clean negative

def extract_patches(acne04_split, patch_split):
    pos_dir = PATCH_DIR / patch_split / "acne"
    neg_dir = PATCH_DIR / patch_split / "no_acne"
    pos_dir.mkdir(parents=True, exist_ok=True)
    neg_dir.mkdir(parents=True, exist_ok=True)

    with open(DATA_DIR / acne04_split / "_annotations.coco.json") as f:
        coco = json.load(f)

    ann_map = {}
    for ann in coco["annotations"]:
        ann_map.setdefault(ann["image_id"], []).append(ann)

    pos_count = neg_count = skipped = 0

    for meta in coco["images"]:
        img_id = meta["id"]
        anns   = ann_map.get(img_id, [])
        if not anns:
            continue

        img    = Image.open(DATA_DIR / acne04_split / meta["file_name"]).convert("RGB")
        img_w, img_h = img.size

        gt_boxes = []  # collect valid gt boxes in x1y1x2y2

        # ── Positive patches ──────────────────────────────────────────────
        for ann in anns:
            x, y, w, h = ann["bbox"]
            if w < MIN_BOX or h < MIN_BOX:
                skipped += 1
                continue
            x1, y1 = int(x), int(y)
            x2, y2 = min(int(x + w), img_w), min(int(y + h), img_h)
            gt_boxes.append([x1, y1, x2, y2])
            patch = img.crop((x1, y1, x2, y2)).resize(
                (PATCH_SIZE, PATCH_SIZE), Image.BILINEAR)
            patch.save(pos_dir / f"{img_id}_{ann['id']}.jpg", quality=90)
            pos_count += 1

        # ── Negative patches — one per positive in this image ─────────────
        for i in range(len(gt_boxes)):
            box = sample_negative(img_w, img_h, gt_boxes, NEG_SIZE)
            if box is None:
                continue
            x1, y1, x2, y2 = box
            patch = img.crop((x1, y1, x2, y2)).resize(
                (PATCH_SIZE, PATCH_SIZE), Image.BILINEAR)
            patch.save(neg_dir / f"{img_id}_neg{i}.jpg", quality=90)
            neg_count += 1

    print(f"[{acne04_split}]  acne={pos_count}  no_acne={neg_count}  skipped(tiny)={skipped}")
    return pos_count, neg_count

print("Starting patch extraction...")
for acne04_split, patch_split in SPLITS.items():
    extract_patches(acne04_split, patch_split)
print("\nDone. Patches saved to data/patches/")

In [ ]:
# Verify counts and show sample patches
import matplotlib.pyplot as plt
import random

%matplotlib inline

for split in ["train", "val"]:
    for cls in ["acne", "no_acne"]:
        files = list((PATCH_DIR / split / cls).glob("*.jpg"))
        print(f"  data/patches/{split}/{cls}: {len(files)} images")

print()

# Show 4 acne + 4 no_acne samples from train
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for row, cls in enumerate(["acne", "no_acne"]):
    files = random.sample(list((PATCH_DIR / "train" / cls).glob("*.jpg")), 4)
    for col, f in enumerate(files):
        axes[row][col].imshow(Image.open(f))
        axes[row][col].set_title(cls, fontsize=9)
        axes[row][col].axis("off")
plt.suptitle("Sample patches — train set", fontsize=13)
plt.tight_layout()
plt.show()